# LeetCode 309 — Best Time to Buy and Sell Stock with Cooldown bằng Python

Link đề: https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-cooldown/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, State Machine

Skeleton LeetCode:

```python
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Đọc đúng đề **mua bán cổ phiếu nhiều lần có thời gian nghỉ (cooldown)**: sau khi **bán**, ngày kế tiếp **không được mua**; tại mỗi thời điểm chỉ giữ tối đa một cổ phiếu.
2. Đi từ **đệ quy vét cạn (mũ)** → **đệ quy có nhớ `dfs(i, đang_giữ)`** → **DP bảng từ dưới lên** → **máy trạng thái 3 trạng thái (hold / sold / rest)** → **tối ưu bộ nhớ `O(1)`**.
3. Biết **mô hình hóa bài toán bằng máy trạng thái**: viết các trạng thái, các cạnh chuyển và giá trị (lãi) trên mỗi cạnh, rồi lấy `max` trên các cạnh đi vào.
4. Phân biệt với các bài họ hàng: 121 (một giao dịch), 122 (không giới hạn, không cooldown), 123/188 (giới hạn số giao dịch), 714 (phí giao dịch).
5. Tránh lỗi kinh điển: quên cooldown (mua ngay sau khi bán), khởi tạo `hold = 0` (cho mua miễn phí), chỉ trả `sold` mà quên `rest`, và index `i + 2` vượt biên.

## 2. Đề bài & ví dụ

Cho mảng `prices` với `prices[i]` là giá cổ phiếu ở ngày `i`. Tìm **lợi nhuận tối đa**. Được thực hiện **nhiều giao dịch** (mua một lần, bán một lần, lặp lại) với ràng buộc:

- Sau khi **bán** cổ phiếu, bạn **không được mua** vào ngày hôm sau (cooldown một ngày).
- Không được tham gia nhiều giao dịch cùng lúc (phải bán trước khi mua lại).

**Ví dụ 1:** `prices = [1,2,3,0,2]` → `3`. Giải thích: `buy, sell, cooldown, buy, sell` (mua ngày 0 giá 1, bán ngày 1 giá 2, nghỉ ngày 2, mua ngày 3 giá 0, bán ngày 4 giá 2: `1 + 2 = 3`).

**Ví dụ 2:** `prices = [1]` → `0`.

**Ràng buộc:** `1 <= prices.length <= 5000`; `0 <= prices[i] <= 1000`.

In [1]:
from typing import List
import random
import sys
from functools import lru_cache

examples = [([1, 2, 3, 0, 2], 3), ([1], 0)]
for p, e in examples:
    print("prices =", p, "-> kỳ vọng", e)

prices = [1, 2, 3, 0, 2] -> kỳ vọng 3
prices = [1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Trạng thái mỗi ngày: "đang giữ" hay "không giữ"

Mỗi ngày bạn ở một trong hai thế: **đang giữ** một cổ phiếu hoặc **không giữ**. Hành động trong ngày: nếu không giữ có thể **mua** (trả `prices[i]`) hoặc **đứng yên**; nếu đang giữ có thể **bán** (nhận `prices[i]`) hoặc **đứng yên**. Cooldown nói rằng sau **bán** thì ngày kế **bắt buộc đứng yên** (không giữ).

In [2]:
# Một lịch hành động hợp lệ cho [1,2,3,0,2] và lãi của nó
prices = [1, 2, 3, 0, 2]
actions = ["buy", "sell", "rest", "buy", "sell"]
profit, holding = 0, False
for price, act in zip(prices, actions):
    if act == "buy":
        assert not holding
        profit -= price
        holding = True
    elif act == "sell":
        assert holding
        profit += price
        holding = False
    print(f"giá {price}: {act:4} -> lãi hiện tại {profit}, đang giữ: {holding}")
print("lãi cuối:", profit)

giá 1: buy  -> lãi hiện tại -1, đang giữ: True
giá 2: sell -> lãi hiện tại 1, đang giữ: False
giá 3: rest -> lãi hiện tại 1, đang giữ: False
giá 0: buy  -> lãi hiện tại 1, đang giữ: True
giá 2: sell -> lãi hiện tại 3, đang giữ: False
lãi cuối: 3


### 3.2. Máy trạng thái hữu hạn (FSM)

Ta tách "không giữ" thành hai trạng thái để diễn tả cooldown:

- **`hold`**: cuối ngày đang giữ cổ phiếu;
- **`sold`**: cuối ngày **vừa bán** hôm nay (ngày mai bị cooldown);
- **`rest`**: cuối ngày không giữ, **và không phải vừa bán hôm nay** (ngày mai có thể mua).

Các cạnh chuyển từ ngày `i − 1` sang ngày `i` (giá `p = prices[i]`):

- `hold[i] = max(hold[i−1], rest[i−1] − p)` — tiếp tục giữ, hoặc mua từ trạng thái `rest`;
- `sold[i] = hold[i−1] + p` — bán cổ phiếu đang giữ;
- `rest[i] = max(rest[i−1], sold[i−1])` — đứng yên; từ `sold` hôm qua chuyển sang `rest` (đã hết cooldown).

Không có cạnh `sold → hold`: đó chính là cooldown.

In [3]:
edges = [
    ("hold", "hold", "đứng yên, giữ tiếp", "0"),
    ("rest", "hold", "mua", "-p"),
    ("hold", "sold", "bán", "+p"),
    ("sold", "rest", "nghỉ cooldown", "0"),
    ("rest", "rest", "đứng yên", "0"),
]
for a, b, why, w in edges:
    print(f"{a:>4} -> {b:<4} | {why:<20} | lãi {w}")
print("không có cạnh sold -> hold  (đó là cooldown)")

hold -> hold | đứng yên, giữ tiếp   | lãi 0
rest -> hold | mua                  | lãi -p
hold -> sold | bán                  | lãi +p
sold -> rest | nghỉ cooldown        | lãi 0
rest -> rest | đứng yên             | lãi 0
không có cạnh sold -> hold  (đó là cooldown)


### 3.3. Đệ quy có nhớ với `lru_cache`

`dfs(i, holding)` = lãi lớn nhất có thể từ ngày `i` trở đi, nếu `holding` cho biết đang giữ hay không. Số trạng thái `2n`, mỗi trạng thái `O(1)` chuyển, nên tổng `O(n)`. Đây là dạng "top-down" tương đương bảng DP.

In [4]:
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print([fib(i) for i in range(10)], "| lần gọi thực sự được nhớ:", fib.cache_info().currsize)

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34] | lần gọi thực sự được nhớ: 10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `prices = [1,2,3,0,2]`, thử vài lịch:

| lịch | lãi |
|---|---|
| mua 1, bán 3 (ngày 0 → 2); nghỉ ngày 3; mua ngày 4? vô ích | `3 − 1 = 2` |
| mua 1, bán 2 (ngày 0 → 1); nghỉ ngày 2; mua 0 (ngày 3), bán 2 (ngày 4) | `1 + 2 = 3` |
| mua 1, bán 2; mua 2 ngay ngày 1 (vi phạm cooldown), bán 3; ... | **không hợp lệ** |

Như vậy "chốt lời ngay khi giá còn tăng" (cách tham lam của bài 122) cho `4` nhưng bị cooldown chặn nên chỉ được `3`. Phải **quyết định có chấp nhận nghỉ một ngày** để đổi lấy mua rẻ hơn.

Ý chính: tại mỗi ngày quan trọng là **trạng thái cuối ngày** (3 giá trị), không cần nhớ toàn bộ lịch sử.

In [5]:
# Lãi nếu bỏ qua cooldown (bài 122) và lãi thật
prices = [1, 2, 3, 0, 2]
no_cooldown = sum(max(0, b - a) for a, b in zip(prices, prices[1:]))
print("lãi khi KHÔNG có cooldown:", no_cooldown, "| lãi đúng đề (có cooldown): 3")

lãi khi KHÔNG có cooldown: 4 | lãi đúng đề (có cooldown): 3


## 5. Cách 1 — Brute Force / baseline: đệ quy vét cạn

**Ý tưởng.** Tại ngày `i`, thử mọi hành động hợp lệ: nếu không giữ, **đứng yên** sang `i + 1` hoặc **mua** rồi sang `i + 1` ở trạng thái giữ; nếu đang giữ, **đứng yên** hoặc **bán** rồi nhảy sang **`i + 2`** (ngày `i + 1` là cooldown). Trả max, **không nhớ**: `O(2ⁿ)`.

In [6]:
def max_profit_brute(prices: List[int]) -> int:
    n = len(prices)

    def go(i, holding):
        if i >= n:
            return 0
        best = go(i + 1, holding)                       # đứng yên
        if holding:
            best = max(best, prices[i] + go(i + 2, False))   # bán, ngày i + 1 cooldown
        else:
            best = max(best, -prices[i] + go(i + 1, True))   # mua
        return best

    return go(0, False)


for p, e in examples:
    assert max_profit_brute(p) == e
print("PASS ví dụ (brute force)")

PASS ví dụ (brute force)


**Độ phức tạp (Cách 1).** Mỗi ngày có tới 2 nhánh nên `O(2ⁿ)`; Space `O(n)` ngăn xếp đệ quy. Dùng làm **oracle** cho `n ≤ 14`.

**Nút thắt.** Hàm chỉ phụ thuộc `(i, holding)`: chỉ `2n` trạng thái khác nhau, nhưng đệ quy tính lại hàng triệu lần. Cần **nhớ kết quả**.

## 6. Ý tưởng thuật toán / bước đột phá

`go(i, holding)` chỉ phụ thuộc hai số, không phụ thuộc lịch sử trước đó. Vậy:

1. **Nhớ** `go(i, holding)` (memoization): `2n` trạng thái × `O(1)` = **`O(n)`**.
2. Viết lại thành **bảng DP** từ dưới lên hoặc từ trên xuống (3 mảng `hold/sold/rest`).
3. Mỗi ngày chỉ cần ngày hôm trước ⇒ **3 biến** ⇒ `O(1)` space.

Tư duy tổng quát: **bài chuỗi quyết định + ràng buộc cục bộ = máy trạng thái**; thêm trạng thái để "mã hóa" ràng buộc (ở đây là `sold` để chặn mua ngày kế).

In [7]:
# Đếm số lần gọi: brute force bùng nổ, có nhớ thì tuyến tính
def count_calls_brute(n):
    calls = 0
    def go(i, holding):
        nonlocal calls
        calls += 1
        if i >= n:
            return 0
        go(i + 1, holding)
        if holding:
            go(i + 2, False)
        else:
            go(i + 1, True)
    go(0, False)
    return calls

for n in (5, 10, 15, 20):
    print(f"n = {n:2}: brute force gọi {count_calls_brute(n):>7} lần, memo chỉ cần tối đa {2 * n} trạng thái")

n =  5: brute force gọi      41 lần, memo chỉ cần tối đa 10 trạng thái
n = 10: brute force gọi     701 lần, memo chỉ cần tối đa 20 trạng thái
n = 15: brute force gọi   11683 lần, memo chỉ cần tối đa 30 trạng thái
n = 20: brute force gọi  194457 lần, memo chỉ cần tối đa 40 trạng thái


## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích.** Giữ nguyên `go(i, holding)` nhưng thêm bộ nhớ. Dùng `lru_cache`, hoặc dict thủ công. Với `n = 5000` đệ quy sâu `5000` dễ tràn ngăn xếp của Python, nên ta **làm ấm bộ nhớ từ cuối về đầu** (gọi `dfs(i, ·)` với `i` giảm dần: mỗi lần chỉ cần tra bộ nhớ, không đệ quy sâu).

In [8]:
def max_profit_memo(prices: List[int]) -> int:
    n = len(prices)
    sys.setrecursionlimit(20000)

    @lru_cache(maxsize=None)
    def dfs(i, holding):
        if i >= n:
            return 0
        best = dfs(i + 1, holding)
        if holding:
            best = max(best, prices[i] + dfs(i + 2, 0))
        else:
            best = max(best, -prices[i] + dfs(i + 1, 1))
        return best

    for i in range(n - 1, -1, -1):                 # làm ấm bộ nhớ từ cuối về đầu: mỗi lần gọi chỉ đệ quy sâu O(1)
        dfs(i, 0)
        dfs(i, 1)
    return dfs(0, 0)


for p, e in examples:
    assert max_profit_memo(p) == e
print("PASS ví dụ (memo)")

PASS ví dụ (memo)


**Độ phức tạp Cách 2:** Time `O(n)`; Space `O(n)` (bộ nhớ + ngăn xếp).

### Cách 3 — DP bảng từ dưới lên (suffix DP)

**Giải thích.** `f[i][0]` = lãi tối đa từ ngày `i` khi **không giữ**; `f[i][1]` = khi **đang giữ**. `f[i][0] = max(f[i+1][0], −p + f[i+1][1])`; `f[i][1] = max(f[i+1][1], p + f[i+2][0])`. Đệm hai hàng `0` ở cuối (`f[n]`, `f[n+1]`) để `i + 2` không vượt biên.

In [9]:
def max_profit_table(prices: List[int]) -> int:
    n = len(prices)
    f = [[0, 0] for _ in range(n + 2)]            # f[n], f[n + 1] = 0: hết ngày, không còn lãi
    for i in range(n - 1, -1, -1):
        p = prices[i]
        f[i][0] = max(f[i + 1][0], -p + f[i + 1][1])      # không giữ: đứng yên hoặc mua
        f[i][1] = max(f[i + 1][1], p + f[i + 2][0])       # đang giữ: đứng yên hoặc bán (cooldown: nhảy 2)
    return f[0][0]


for p, e in examples:
    assert max_profit_table(p) == e
print("PASS ví dụ (bảng DP)")

PASS ví dụ (bảng DP)


**Độ phức tạp Cách 3:** Time `O(n)`; Space `O(n)`, không đệ quy nên chạy tốt với `n = 5000`.

### Cách 4 — Máy trạng thái 3 trạng thái (hold / sold / rest) theo từng ngày

**Giải thích.** Đi từ trái sang phải với ba mảng. Khởi tạo ngày 0: `hold = −prices[0]` (đã mua), `sold = −∞` (chưa thể bán), `rest = 0`. Mỗi ngày tính theo công thức ở 3.2. Đáp án `max(sold, rest)` ở ngày cuối (không giữ cổ phiếu).

In [10]:
def max_profit_states(prices: List[int]) -> int:
    n = len(prices)
    NEG = float("-inf")
    hold = [NEG] * n
    sold = [NEG] * n
    rest = [NEG] * n
    hold[0], sold[0], rest[0] = -prices[0], NEG, 0
    for i in range(1, n):
        p = prices[i]
        hold[i] = max(hold[i - 1], rest[i - 1] - p)
        sold[i] = hold[i - 1] + p
        rest[i] = max(rest[i - 1], sold[i - 1])
    return max(sold[-1], rest[-1])


for p, e in examples:
    assert max_profit_states(p) == e
print("PASS ví dụ (3 trạng thái)")

PASS ví dụ (3 trạng thái)


**Độ phức tạp Cách 4:** Time `O(n)`; Space `O(n)` (ba mảng).

### Cách 5 — Máy trạng thái tối ưu `O(1)` bộ nhớ (cách nộp bài)

**Giải thích.** Mỗi ngày chỉ cần ngày trước, nên dùng ba biến `hold, sold, rest`, cập nhật **cùng lúc** (cẩn thận dùng bản cũ của `hold`/`sold`). Một biến thể gọn hơn của lời giải tham khảo (memo `dfs(i, j)`) nhưng chạy lặp nên không lo đệ quy sâu.

In [11]:
def max_profit_o1(prices: List[int]) -> int:
    hold, sold, rest = -prices[0], float("-inf"), 0
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)


for p, e in examples:
    assert max_profit_o1(p) == e
print("PASS ví dụ (O(1) space)")

PASS ví dụ (O(1) space)


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(1)`.

> Cách 5 là cách nộp bài: ngắn, nhanh. Cách 2 và 3 giúp thấy rõ từ đệ quy sang bảng. Cách 1 dùng làm oracle.

## 8. TRACE từng bước

Chạy máy trạng thái trên `prices = [1,2,3,0,2]` và in `hold / sold / rest` từng ngày.

In [12]:
def trace(prices):
    NEG = float("-inf")
    hold, sold, rest = -prices[0], NEG, 0
    print(f"ngày 0 (giá {prices[0]}): hold={hold}, sold={sold}, rest={rest}")
    for i in range(1, len(prices)):
        p = prices[i]
        new_hold = max(hold, rest - p)
        new_sold = hold + p
        new_rest = max(rest, sold)
        print(f"ngày {i} (giá {p}): hold = max({hold}, {rest}-{p}) = {new_hold}; "
              f"sold = {hold}+{p} = {new_sold}; rest = max({rest}, {sold}) = {new_rest}")
        hold, sold, rest = new_hold, new_sold, new_rest
    ans = max(sold, rest)
    print("đáp án = max(sold, rest) =", ans)
    return ans


assert trace([1, 2, 3, 0, 2]) == 3

ngày 0 (giá 1): hold=-1, sold=-inf, rest=0
ngày 1 (giá 2): hold = max(-1, 0-2) = -1; sold = -1+2 = 1; rest = max(0, -inf) = 0
ngày 2 (giá 3): hold = max(-1, 0-3) = -1; sold = -1+3 = 2; rest = max(0, 1) = 1
ngày 3 (giá 0): hold = max(-1, 1-0) = 1; sold = -1+0 = -1; rest = max(1, 2) = 2
ngày 4 (giá 2): hold = max(1, 2-2) = 1; sold = 1+2 = 3; rest = max(2, -1) = 2
đáp án = max(sold, rest) = 3


Bảng tóm tắt:

| ngày | giá | hold | sold | rest |
|---|---|---|---|---|
| 0 | 1 | −1 | −∞ | 0 |
| 1 | 2 | −1 | 1 | 0 |
| 2 | 3 | −1 | 2 | 1 |
| 3 | 0 | 1 | −1 | 2 |
| 4 | 2 | 1 | 3 | 2 |

Ngày 4: `max(sold, rest) = max(3, 2) = 3`. Ở ngày 3, `hold = rest₂ − 0 = 1` vì ngày 2 mới `rest = max(0, sold₁ = 1) = 1` (đã qua cooldown).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 5 (`O(n)` thời gian, `O(1)` bộ nhớ), đúng chữ ký của LeetCode.

In [13]:
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        hold, sold, rest = -prices[0], float("-inf"), 0
        for p in prices[1:]:
            hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
        return max(sold, rest)


for p, e in examples:
    assert Solution().maxProfit(p) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Đầy đủ trạng thái:** mỗi ngày kết thúc ở đúng một trong ba trạng thái `hold`, `sold`, `rest`; mọi lịch hợp lệ tương ứng một đường đi trong máy trạng thái.
- **Cooldown được mã hóa:** không có cạnh `sold → hold`; muốn mua phải qua `rest`, tức là ít nhất một ngày sau khi bán.
- **Quy hoạch động tối ưu con:** giá trị lớn nhất để kết thúc ngày `i` ở trạng thái `s` bằng max của (giá trị lớn nhất ngày `i − 1` ở trạng thái trước `s`) + lãi cạnh. Vì lãi cạnh chỉ phụ thuộc giá ngày `i`, đây là bài toán đường đi dài nhất trong DAG và tối ưu cục bộ cho tối ưu toàn cục.
- **Đáp án:** ngày cuối không được còn cổ phiếu (nếu còn thì chưa chốt lãi và luôn kém hơn bán hoặc không mua), nên lấy `max(sold, rest)`; `hold` không bao giờ tốt hơn.
- **Khởi tạo:** `sold[0] = −∞` vì không thể bán ngày đầu; `hold[0] = −prices[0]`; `rest[0] = 0`.

## 11. Phân tích Time / Space Complexity

- **Cách 1 (vét cạn):** `O(2ⁿ)` time, `O(n)` stack.
- **Cách 2 (memo):** `O(n)` time, `O(n)` space (cả đệ quy sâu `n`).
- **Cách 3 (bảng suffix):** `O(n)` time, `O(n)` space.
- **Cách 4 (3 mảng):** `O(n)` time, `O(n)` space.
- **Cách 5 (3 biến):** `O(n)` time, **`O(1)`** space.
- Với `n = 5000`: Cách 3, 4, 5 chỉ cần ~`10⁴` thao tác.

## 12. Bộ test

Gom các cách vào từ điển; đối chiếu với **brute force** (cho `n ≤ 14`) và giữa các cách nhanh với nhau (cho `n` lớn).

In [14]:
impls = {
    "brute": max_profit_brute,
    "memo": max_profit_memo,
    "table": max_profit_table,
    "states": max_profit_states,
    "o1": max_profit_o1,
    "submit": lambda prices: Solution().maxProfit(prices),
}
FAST = ("memo", "table", "states", "o1", "submit")


def check(prices, expected=None, names=None):
    base = max_profit_brute(prices) if len(prices) <= 14 else max_profit_o1(prices)
    if expected is not None:
        assert base == expected, (prices, base, expected)
    for name, fn in impls.items():
        if names is not None and name not in names:
            continue
        if name == "brute" and len(prices) > 14:
            continue
        assert fn(list(prices)) == base, (name, prices, fn(list(prices)), base)
    return base


check([1, 2, 3, 0, 2], 3)
check([1], 0)
print("PASS 2 ví dụ của đề")

PASS 2 ví dụ của đề


### 12.1. Các trường hợp biên và giá trị tính tay

In [15]:
cases = [
    ([2, 1], 0, "giảm, không nên mua"),
    ([1, 2], 1, "mua rồi bán"),
    ([1, 2, 4], 3, "mua 1 bán 4"),
    ([1, 2, 3, 4, 5], 4, "tăng đều: mua đầu bán cuối"),
    ([5, 4, 3], 0, "giảm đều"),
    ([2, 1, 4], 3, "mua 1 bán 4"),
    ([6, 1, 3, 2, 4, 7], 6, "mua 1 bán 7"),
    ([0, 0, 0, 0], 0, "toàn 0"),
    ([1000, 0, 1000], 1000, "biên giá: mua 0 bán 1000"),
    ([3, 3, 3], 0, "giá không đổi"),
]
for p, e, why in cases:
    assert check(p, e) == e
    print("PASS", p, "->", e, "|", why)

PASS [2, 1] -> 0 | giảm, không nên mua
PASS [1, 2] -> 1 | mua rồi bán
PASS [1, 2, 4] -> 3 | mua 1 bán 4
PASS [1, 2, 3, 4, 5] -> 4 | tăng đều: mua đầu bán cuối
PASS [5, 4, 3] -> 0 | giảm đều
PASS [2, 1, 4] -> 3 | mua 1 bán 4
PASS [6, 1, 3, 2, 4, 7] -> 6 | mua 1 bán 7
PASS [0, 0, 0, 0] -> 0 | toàn 0
PASS [1000, 0, 1000] -> 1000 | biên giá: mua 0 bán 1000
PASS [3, 3, 3] -> 0 | giá không đổi


### 12.2. Tính chất và phản ví dụ

In [16]:
random.seed(309)

# Tính chất 1: kết quả luôn >= 0, và <= lãi khi KHÔNG có cooldown (122)
# Tính chất 2: giá đơn điệu giảm -> lãi 0
# Tính chất 3: cộng hằng số vào mọi giá không đổi kết quả
for _ in range(300):
    n = random.randint(1, 25)
    p = [random.randint(0, 30) for _ in range(n)]
    r = max_profit_o1(p)
    free = sum(max(0, b - a) for a, b in zip(p, p[1:]))
    assert 0 <= r <= free
    assert max_profit_o1([x + 7 for x in p]) == r
    assert max_profit_o1(sorted(p, reverse=True)) == 0
print("PASS tính chất: 0 <= lãi <= lãi không cooldown; bất biến khi cộng hằng; giảm dần -> 0")

# Phản ví dụ: cooldown thực sự làm giảm lãi
p = [1, 2, 3, 0, 2]
assert max_profit_o1(p) == 3 and sum(max(0, b - a) for a, b in zip(p, p[1:])) == 4
print("PASS phản ví dụ: [1,2,3,0,2] lãi 3 < 4 (không cooldown)")

PASS tính chất: 0 <= lãi <= lãi không cooldown; bất biến khi cộng hằng; giảm dần -> 0
PASS phản ví dụ: [1,2,3,0,2] lãi 3 < 4 (không cooldown)


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [17]:
random.seed(2026)
for trial in range(1500):
    n = random.randint(1, 13)
    hi = random.choice([2, 5, 20, 1000])
    check([random.randint(0, hi) for _ in range(n)])

for trial in range(40):                                   # n lớn: chỉ so các cách nhanh với nhau
    n = random.randint(500, 5000)
    check([random.randint(0, 1000) for _ in range(n)], names=("table", "states", "o1", "submit"))

# n = 5000: kiểm tra riêng memo một lần
big = [random.randint(0, 1000) for _ in range(5000)]
assert max_profit_memo(big) == max_profit_o1(big)
print("PASS 1500 test nhỏ (với brute), 40 test lớn (so các cách nhanh), 1 test n = 5000 (memo)")

PASS 1500 test nhỏ (với brute), 40 test lớn (so các cách nhanh), 1 test n = 5000 (memo)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên cooldown (mua ngay sau khi bán)

In [18]:
def sai_khong_cooldown(prices):
    hold, rest = -prices[0], 0
    for p in prices[1:]:
        hold, rest = max(hold, rest - p), max(rest, hold + p)     # sai: sau khi bán đã được mua ngay
    return rest


print("không cooldown ->", sai_khong_cooldown([1, 2, 3, 0, 2]), "| đúng: 3")
assert sai_khong_cooldown([1, 2, 3, 0, 2]) != 3

không cooldown -> 4 | đúng: 3


**Vì sao sai:** bản này mua-bán không giới hạn nên ra `4` (mua 1 bán 2, mua 2 bán 3, mua 0 bán 2: `1 + 1 + 2`). Cooldown cấm mua vào ngày ngay sau khi bán: cần trạng thái `sold` riêng.

### Lỗi 2: Khởi tạo `hold = 0`

In [19]:
def sai_hold_bang_0(prices):
    hold, sold, rest = 0, float("-inf"), 0           # sai: "đang giữ" mà chưa tốn tiền
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)


print("hold = 0 ->", sai_hold_bang_0([1, 2, 3, 0, 2]), "| đúng: 3")
assert sai_hold_bang_0([1, 2, 3, 0, 2]) != 3

hold = 0 -> 4 | đúng: 3


**Vì sao sai:** `hold[0]` phải là `−prices[0]` vì đã trả tiền mua. Đặt `0` cho phép "nhận" cổ phiếu miễn phí nên lãi ảo.

### Lỗi 3: Chỉ trả `sold`, quên `rest`

In [20]:
def sai_chi_tra_sold(prices):
    hold, sold, rest = -prices[0], float("-inf"), 0
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return sold                                      # sai: ngày cuối có thể ở trạng thái rest


bad_input = [1, 2, 3, 0, 2, 1]
print("chỉ sold ->", sai_chi_tra_sold(bad_input), "| đúng:", max_profit_o1(bad_input))
assert sai_chi_tra_sold(bad_input) != max_profit_o1(bad_input)

chỉ sold -> 2 | đúng: 3


**Vì sao sai:** ngày cuối có thể đã bán từ trước và đang ở `rest`; chỉ lấy `sold` bỏ lỡ trường hợp đó (ở ví dụ ngày cuối giá giảm nên chốt lãi sớm hơn tốt hơn).

### Lỗi 4: Cập nhật trạng thái "tuần tự" thay vì đồng thời

In [21]:
def sai_cap_nhat_tuan_tu(prices):
    hold, sold, rest = -prices[0], float("-inf"), 0
    for p in prices[1:]:
        hold = max(hold, rest - p)
        sold = hold + p                              # sai: dùng hold MỚI (vừa mua hôm nay rồi bán luôn hôm nay)
        rest = max(rest, sold)
    return max(sold, rest)


print("cập nhật tuần tự ->", sai_cap_nhat_tuan_tu([1, 2, 3, 0, 2]), "| đúng: 3")
assert sai_cap_nhat_tuan_tu([1, 2, 3, 0, 2]) != 3

cập nhật tuần tự -> 4 | đúng: 3


**Vì sao sai:** mỗi cạnh phải dùng giá trị của **ngày hôm trước**. Cập nhật tuần tự cho phép mua và bán trong cùng một ngày (lãi 0 nên chưa hại) nhưng cũng làm `rest` nhận `sold` của hôm nay (bỏ qua cooldown). Python tuple assignment `a, b, c = ..., ..., ...` tính vế phải bằng giá trị cũ nên an toàn.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Mua bán cổ phiếu + ràng buộc (cooldown, phí, số giao dịch)" → **DP máy trạng thái**: liệt kê trạng thái cuối ngày, vẽ cạnh, viết `max`.
- Ràng buộc "sau X phải nghỉ" → thêm một trạng thái "vừa làm X".
- Nhớ bảng họ hàng: 121 (1 lần), 122 (không giới hạn), 123 (tối đa 2), 188 (tối đa k), 309 (cooldown), 714 (phí).
- Dạng memo `dfs(i, giữ)` dễ nghĩ; chuyển sang bảng suffix để tránh giới hạn đệ quy Python.
- Cập nhật nhiều biến phụ thuộc nhau: dùng **phép gán tuple** để dùng giá trị cũ.
- Khởi tạo: trạng thái bất khả thi dùng `−∞`.

## 15. Biến thể của bài

1. **Cooldown `c` ngày** (tổng quát): sau khi bán phải nghỉ `c` ngày.
2. **Phí giao dịch (LeetCode 714):** mỗi lần bán trừ `fee`.
3. **Tối đa `k` giao dịch (LeetCode 188).**
4. **Tính cả lịch hành động** (in ra các ngày mua/bán).
5. **Không cooldown (LeetCode 122):** cộng các hiệu dương liên tiếp.

In [22]:
def max_profit_cooldown_c(prices, c=1):
    """Biến thể 1: sau khi bán phải nghỉ c ngày (c = 1 là bài gốc). DP suffix."""
    n = len(prices)
    f = [[0, 0] for _ in range(n + c + 2)]
    for i in range(n - 1, -1, -1):
        p = prices[i]
        f[i][0] = max(f[i + 1][0], -p + f[i + 1][1])
        f[i][1] = max(f[i + 1][1], p + f[min(i + 1 + c, n)][0])
    return f[0][0]


def max_profit_fee(prices, fee):
    """Biến thể 2 (714): phí mỗi giao dịch."""
    hold, cash = -prices[0], 0
    for p in prices[1:]:
        hold, cash = max(hold, cash - p), max(cash, hold + p - fee)
    return cash


def max_profit_k(prices, k):
    """Biến thể 3 (188): tối đa k giao dịch."""
    if not prices or k == 0:
        return 0
    buy = [-prices[0]] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices[1:]:
        for j in range(1, k + 1):
            buy[j] = max(buy[j], sell[j - 1] - p)
            sell[j] = max(sell[j], buy[j] + p)
    return sell[k]


def best_actions(prices):
    """Biến thể 4: trả (lãi, danh sách hành động) bằng bảng suffix + truy vết."""
    n = len(prices)
    f = [[0, 0] for _ in range(n + 2)]
    for i in range(n - 1, -1, -1):
        p = prices[i]
        f[i][0] = max(f[i + 1][0], -p + f[i + 1][1])
        f[i][1] = max(f[i + 1][1], p + f[i + 2][0])
    acts, i, h = [], 0, 0
    while i < n:
        p = prices[i]
        if h == 0:
            if -p + f[i + 1][1] > f[i + 1][0]:
                acts.append((i, "buy"))
                h = 1
            i += 1
        else:
            if p + f[i + 2][0] > f[i + 1][1]:
                acts.append((i, "sell"))
                h = 0
                i += 2
            else:
                i += 1
    return f[0][0], acts


def max_profit_free(prices):
    """Biến thể 5 (122)."""
    return sum(max(0, b - a) for a, b in zip(prices, prices[1:]))


random.seed(15)
for _ in range(300):
    p = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    assert max_profit_cooldown_c(p, 1) == max_profit_o1(p)
    assert max_profit_cooldown_c(p, 0) == max_profit_free(p)
    assert max_profit_fee(p, 0) == max_profit_free(p)
    assert max_profit_k(p, len(p)) == max_profit_free(p)
    assert max_profit_cooldown_c(p, 2) <= max_profit_cooldown_c(p, 1)
    gain, acts = best_actions(p)
    assert gain == max_profit_o1(p)
    total, last_sell = 0, -10
    for day, a in acts:
        if a == "buy":
            assert day > last_sell + 1 or last_sell < 0 or day >= last_sell + 2
            total -= p[day]
        else:
            total += p[day]
            last_sell = day
    assert total == gain
assert max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8
assert max_profit_k([3, 2, 6, 5, 0, 3], 2) == 7
print("PASS biến thể: cooldown c, phí, tối đa k, truy vết, không cooldown")

PASS biến thể: cooldown c, phí, tối đa k, truy vết, không cooldown


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi hành động | `O(2ⁿ)` | `O(n)` | oracle, `n ≤ 14` |
| 2. Memo | `dfs(i, giữ)` có nhớ | `O(n)` | `O(n)` | dễ nghĩ, cẩn thận đệ quy sâu |
| 3. Bảng suffix | DP từ cuối về đầu | `O(n)` | `O(n)` | an toàn, không đệ quy |
| 4. 3 mảng trạng thái | hold/sold/rest | `O(n)` | `O(n)` | dễ truy vết |
| 5. 3 biến | tối ưu bộ nhớ | `O(n)` | `O(1)` | **cách nộp bài** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `max_profit_o1([2, 1])`, `max_profit_o1([1, 2, 4])`, `max_profit_o1([6, 1, 3, 2, 4, 7])`.

**Bài 2 (tính tay).** Với `prices = [1, 4, 2]`: lập bảng `hold / sold / rest` từng ngày và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `max_profit_todo` theo Cách 5.

In [23]:
def max_profit_todo(prices):
    # TODO 1: khởi tạo hold = -prices[0], sold = float("-inf"), rest = 0
    # TODO 2: với mỗi giá p từ ngày 1: hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    # TODO 3: trả max(sold, rest)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_profit_memo2(prices)` theo Cách 2 (đệ quy có nhớ `dfs(i, giữ)`), và `max_profit_table2` theo Cách 3.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_profit_fee_cd(prices, fee)` có **cả cooldown một ngày và phí giao dịch `fee`** mỗi lần bán. (b) Viết hàm đếm **số cách** (số lịch hành động khác nhau) đạt lãi tối ưu cho mảng nhỏ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2, 1]` giá giảm; `[1, 2, 4]` mua 1 bán 4. Bài 2: dùng công thức ở 3.2.
- **Mức 2 (vừa):** Bài 1: `0`, `3`, `6`. Bài 2: ngày 0 `(hold, sold, rest) = (−1, −∞, 0)`; ngày 1 (giá 4): `(−1, 3, 0)`; ngày 2 (giá 2): `(max(−1, 0 − 2) = −1, −1 + 2 = 1, max(0, 3) = 3)`; đáp án `max(1, 3) = 3`.
- **Mức 3 (sâu):** Bài 4: `dfs(i, giữ)` trả 0 khi `i >= n`; bán nhảy `i + 2`. Bài 5(a): trừ `fee` trên cạnh bán: `sold = hold + p − fee`. Bài 5(b): đếm đường đi đạt max, dùng cặp (lãi, số cách) trong DP suffix.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert max_profit_o1([2, 1]) == 0
assert max_profit_o1([1, 2, 4]) == 3
assert max_profit_o1([6, 1, 3, 2, 4, 7]) == 6

# Bài 2
assert max_profit_o1([1, 4, 2]) == 3

# Bài 3
def max_profit_todo_ans(prices):
    hold, sold, rest = -prices[0], float("-inf"), 0
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)

# Bài 4
def max_profit_memo2(prices):
    n = len(prices)
    memo = {}

    def dfs(i, holding):
        if i >= n:
            return 0
        key = (i, holding)
        if key in memo:
            return memo[key]
        best = dfs(i + 1, holding)
        if holding:
            best = max(best, prices[i] + dfs(i + 2, 0))
        else:
            best = max(best, -prices[i] + dfs(i + 1, 1))
        memo[key] = best
        return best

    return dfs(0, 0)


def max_profit_table2(prices):
    n = len(prices)
    f = [[0, 0] for _ in range(n + 2)]
    for i in range(n - 1, -1, -1):
        f[i][0] = max(f[i + 1][0], -prices[i] + f[i + 1][1])
        f[i][1] = max(f[i + 1][1], prices[i] + f[i + 2][0])
    return f[0][0]

# Bài 5
def max_profit_fee_cd(prices, fee):
    hold, sold, rest = -prices[0], float("-inf"), 0
    for p in prices[1:]:
        hold, sold, rest = max(hold, rest - p), hold + p - fee, max(rest, sold)
    return max(sold, rest, 0)


def brute_fee_cd(prices, fee):
    n = len(prices)

    def go(i, holding):
        if i >= n:
            return 0
        best = go(i + 1, holding)
        if holding:
            best = max(best, prices[i] - fee + go(i + 2, False))
        else:
            best = max(best, -prices[i] + go(i + 1, True))
        return best

    return go(0, False)


def enumerate_plans(prices):
    """Oracle: liệt kê mọi lịch hợp lệ (dãy hành động), đếm số lịch đạt lãi tối ưu."""
    n = len(prices)
    results = []

    def rec(i, holding, can_buy, profit):
        if i == n:
            if not holding:
                results.append(profit)
            return
        rec(i + 1, holding, True, profit)                       # đứng yên (hết cooldown)
        if holding:
            rec(i + 1, False, False, profit + prices[i])         # bán
        elif can_buy:
            rec(i + 1, True, True, profit - prices[i])           # mua

    rec(0, False, True, 0)
    best = max(results)
    return best, results.count(best)


for p in ([1, 2, 3, 0, 2], [2, 1], [1, 2, 4], [6, 1, 3, 2, 4, 7]):
    assert max_profit_todo_ans(p) == max_profit_memo2(p) == max_profit_table2(p) == max_profit_o1(p)

random.seed(19)
for _ in range(300):
    p = [random.randint(0, 15) for _ in range(random.randint(1, 10))]
    fee = random.randint(0, 4)
    assert max_profit_fee_cd(p, fee) == brute_fee_cd(p, fee), (p, fee)
    best, ways = enumerate_plans(p)
    assert best == max_profit_o1(p)
    assert ways >= 1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (máy trạng thái hold / sold / rest)

```python
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        hold, sold, rest = -prices[0], float("-inf"), 0
        for p in prices[1:]:
            hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
        return max(sold, rest)
```

**Checklist**

- [ ] Đã mô hình hóa 3 trạng thái cuối ngày (đang giữ / vừa bán / nghỉ) và vẽ cạnh?
- [ ] Không có cạnh `sold → hold` (cooldown)?
- [ ] `hold` khởi tạo `−prices[0]`, `sold` khởi tạo `−∞`, `rest` khởi tạo `0`?
- [ ] Cập nhật ba biến bằng gán tuple (dùng giá trị ngày hôm trước)?
- [ ] Trả `max(sold, rest)` (không trả `hold`)?
- [ ] Đã thử `n = 1`, giá giảm đều, giá không đổi, giá biên `0` và `1000`?
- [ ] Đã đối chiếu với brute force trên input ngẫu nhiên có seed?
- [ ] Skeleton đúng `class Solution` / `maxProfit(self, prices: List[int]) -> int`?